In [1]:
# =========================================
# IMPORT LIBRARIES
# =========================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import boto3
from botocore.client import Config

In [2]:
# =========================================
# INIT SPARK SESSION
# =========================================

spark = SparkSession.builder \
    .appName("SV3_Crypto_ETL") \
    .getOrCreate()

print("Spark Started Successfully")

# =========================================
# MINIO S3A CONFIG
# =========================================

hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.access.key", "admin")
hadoop_conf.set("fs.s3a.secret.key", "password123")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

hadoop_conf.set(
    "fs.s3a.aws.credentials.provider",
    "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
)

print("MinIO Configuration Completed")

Spark Started Successfully
MinIO Configuration Completed


In [3]:
# =========================================
# LOAD RAW DATA FROM MINIO
# =========================================

df = spark.read.csv(
    "s3a://crypto-raw-data/bitcoin_1m.csv",
    header=True,
    inferSchema=True
)

print("Raw Dataset Loaded")

print("Total Rows:", df.count())

df.show(20)

df.printSchema()

Raw Dataset Loaded
Total Rows: 1000
+-------------------+--------+--------+--------+--------+-----------+
|          timestamp|    open|    high|     low|   close|     volume|
+-------------------+--------+--------+--------+--------+-----------+
|2026-06-10 16:34:00|61230.55|61252.99|61226.98|61242.91| 0.16060134|
|2026-06-10 16:35:00|61243.79|61284.04| 61241.0| 61284.0| 0.91514885|
|2026-06-10 16:36:00|61297.02|61330.26|61267.21|61267.21| 0.44197633|
|2026-06-10 16:37:00|61267.21|61267.21|61218.99|61253.44| 0.32322552|
|2026-06-10 16:38:00|61254.41| 61373.8|61254.41|61366.47| 0.19285511|
|2026-06-10 16:39:00|61358.49|61367.26|61328.63|61331.74| 0.18446825|
|2026-06-10 16:40:00|61325.65|61325.65|61254.25|61254.25| 0.77831114|
|2026-06-10 16:41:00| 61270.3|61296.65| 61270.3|61274.73| 0.82656264|
|2026-06-10 16:42:00|61274.74|61282.94| 61262.0|61270.01|  0.1508074|
|2026-06-10 16:43:00|61269.86|61269.86|61225.85|61263.17|  0.1555804|
|2026-06-10 16:44:00|61263.18|61263.18|61203.06|61213.